In [ ]:
# SkinNova — Gemma 4 E2B → .litertlm export + checks (generated by ml/llm/notebooks/make_export.py — edit there)
MODE = "stock"


In [ ]:
%%capture
!pip install -q uv
!uv venv /tmp/ve --python 3.12
!uv pip install --python /tmp/ve/bin/python --index-url https://download.pytorch.org/whl/cpu "torch==2.11.0" "torchvision==0.26.0"
!uv pip install --python /tmp/ve/bin/python "litert-torch==0.9.4" "litert-lm-api==0.17.1" "litert-lm-builder==0.17.1" "ai-edge-litert==2.2.0" "ai-edge-quantizer==0.9.0" "litert-converter==0.4.0" "transformers==5.18.0" "tokenizers==0.23.2" "huggingface-hub==1.33.0" "protobuf==7.36.2" "torchao==0.18.0" "safetensors==0.8.0" "sentencepiece==0.2.2" "numpy==2.5.2" "pillow==12.3.0" "jax==0.11.2" "jaxlib==0.11.2" "pydantic" "python-dotenv" "soundfile" "datasets"


In [ ]:
open('/tmp/run.py', 'w').write('"""Runs INSIDE the Kaggle export notebook\'s isolated venv (see make_export.py): export → capabilities → L6-style suite.\nUsage there: /tmp/ve/bin/python /tmp/run.py <stock|merged>"""\nimport glob\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\n\nMODE = sys.argv[1]\nDATA = os.path.dirname(glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)[0])\nsys.path.insert(0, DATA + "/code")\nfrom ml.eval.llm_metrics import ANALYSIS_KEYS, score_analysis, summarize  # noqa: E402\n\nOUT = "/kaggle/working"\nos.makedirs(f"{OUT}/reports", exist_ok=True); os.makedirs(f"{OUT}/export", exist_ok=True)\nREPORT = {"mode": MODE, "steps": []}\n\n\ndef save():\n    json.dump(REPORT, open(f"{OUT}/reports/export_{MODE}.json", "w"), indent=1, default=str)\n\n\nfrom huggingface_hub import hf_hub_download, snapshot_download  # noqa: E402\n\nif MODE == "stock":\n    SRC = snapshot_download("unsloth/gemma-4-E2B-it", local_dir="/tmp/gemma4_e2b")\nelse:\n    SRC = os.path.dirname(glob.glob("/kaggle/input/**/merged/config.json", recursive=True)[0])\nREPORT["source"] = SRC; save(); print("source", SRC, flush=True)\n\n\nimport threading  # noqa: E402\n\nMEM = []\n\n\ndef _memlog():\n    while True:\n        try:\n            info = dict(l.split(":", 1) for l in open("/proc/meminfo").read().splitlines())\n            MEM.append((round(time.time()), int(info["MemTotal"].split()[0]) // 1024, int(info["MemAvailable"].split()[0]) // 1024))\n        except Exception:  # noqa: BLE001\n            pass\n        time.sleep(2)\n\n\nthreading.Thread(target=_memlog, daemon=True).start()\n\n\nPATCHED = os.path.join(os.path.dirname(os.path.abspath(__file__)), "export_patched.py")   # memory fixes, see that file\n\n\ndef export():\n    out = "/tmp/exp"\n    subprocess.run(["rm", "-rf", out])\n    # --export_audio_encoder is read only by the Qwen3 ASR path in litert-torch 0.9.4; Gemma 4 gets no audio either way.\n    cmd = [sys.executable, PATCHED, "export_hf", SRC, out, "--task=image_text_to_text",\n           "--externalize_embedder=True", "--quantization_recipe=dynamic_wi8_emb4_afp32", "--export_vision_encoder=True",\n           "--export_audio_encoder=False", "--bundle_litert_lm=True", "--use_jinja_template=True",\n           "--experimental_lightweight_conversion=True",\n           # LiteRT-LM\'s Jinja engine rejects the HF repo template ("unknown method", export v9) → bundle the template from\n           # Google\'s official .litertlm, which is also the template the LoRA was trained with\n           f"--jinja_chat_template_override={DATA}/code/chat_template_litertlm.jinja"]\n    t = time.time()\n    p = subprocess.run(cmd, capture_output=True, text=True)\n    files = glob.glob(f"{out}/**/*.litertlm", recursive=True)\n    REPORT["steps"].append({"rc": p.returncode, "s": round(time.time() - t), "files": files,\n                            "mem_total_mb": MEM[-1][1] if MEM else None, "mem_min_available_mb": min((m[2] for m in MEM), default=None),\n                            "released": [l for l in p.stdout.splitlines() if l.startswith("[skinnova]")],\n                            "stdout_tail": p.stdout[-1500:], "stderr_tail": p.stderr[-3000:]}); save()\n    print(f"export rc={p.returncode} files={files} ({round(time.time() - t)}s)", flush=True)\n    if p.returncode:\n        print(p.stderr[-2500:], flush=True)\n    return files[0] if files else None\n\n\nmodel = export()\nif not model:\n    raise SystemExit(f"export failed; see reports/export_{MODE}.json")\ndst = f"{OUT}/export/skinnova-e2b-{MODE}.litertlm"\nsubprocess.run(["cp", model, dst])\nh = hashlib.sha256()\nwith open(dst, "rb") as f:\n    for b in iter(lambda: f.read(1 << 24), b""):\n        h.update(b)\nREPORT["litertlm"] = {"path": dst, "bytes": os.path.getsize(dst), "sha256": h.hexdigest()}; save(); print(REPORT["litertlm"], flush=True)\n\nimport litert_lm as L  # noqa: E402\nfrom PIL import Image  # noqa: E402\n\n\ndef caps(path):\n    c = L.Capabilities(path); m = c.input_modalities   # a property in litert-lm-api 0.17.1 (Kotlin: inputModalities())\n    r = {"text": m.text, "vision": m.vision, "audio": m.audio}; c.close()\n    return r\n\n\ndef load_engine(path):\n    cap = caps(path)\n    t = time.time()\n    eng = L.Engine(path, backend=L.Backend.CPU(), vision_backend=L.Backend.CPU() if cap["vision"] else None,\n                   audio_backend=L.Backend.CPU() if cap["audio"] else None, max_num_tokens=4096)\n    return eng, cap, round(time.time() - t, 1)\n\n\ndef ask(eng, parts, system=None, max_tok=700):\n    # greedy (parity with the HF greedy outputs); system message in the app\'s form (one-part list, C API parses JSON)\n    sm = json.dumps([{"type": "text", "text": system}], ensure_ascii=False) if system else None\n    conv = eng.create_conversation(system_message=sm, sampler_config=L.SamplerConfig(top_k=1, top_p=1.0, temperature=0.0, seed=3407),\n                                   thinking_config=L.ThinkingConfig(enable_thinking=False), max_output_tokens=max_tok)\n    try:\n        out = conv.send_message(L.Contents(parts))\n    finally:\n        conv.close()\n    return "".join(c.get("text", "") for c in out.get("content", []) if isinstance(c, dict))\n\n\ndef run_suite(path, tag, cases, n_gray):\n    eng, cap, load_s = load_engine(path)\n    res = {"load_s": load_s, "capabilities": cap, "text_probe": ask(eng, [L.Content.Text("Reply with exactly the word OK.")], max_tok=8)}\n    gray = "/tmp/gray.jpg"; Image.new("RGB", (512, 512), (128, 128, 128)).save(gray)\n    rows = []\n    for i, r in enumerate(cases):\n        parts, kinds = [], []\n        for c in r["messages"][1]["content"]:\n            parts.append(L.Content.ImageFile(f\'{DATA}/{r["image"]}\') if c["type"] == "image" else L.Content.Text(c["text"]))\n            kinds.append(c["type"])\n        t1 = time.time(); txt = ask(eng, parts, r["messages"][0]["content"]); dt = time.time() - t1\n        row = {"id": r["id"], "task": r["task"], "s": round(dt, 1), **score_analysis(r["meta"], txt), "out": txt}\n        if cap["vision"] and i < n_gray:\n            g = ask(eng, [L.Content.ImageFile(gray) if k == "image" else q for q, k in zip(parts, kinds)], r["messages"][0]["content"])\n            row["gray_cats"] = score_analysis(r["meta"], g)["cats"]; row["gray_changed"] = set(row["gray_cats"]) != set(row["cats"])\n        rows.append(row); REPORT[tag] = {**res, "rows": rows}; save()\n        print(tag, r["id"], "valid", row["valid"], "agree", row["cat_agree"], f"{dt:.0f}s", flush=True)\n    res["analysis"] = summarize(rows, ANALYSIS_KEYS); res["rows"] = rows\n    g = [x["gray_changed"] for x in rows if "gray_changed" in x]\n    res["image_ablation_changed"] = (sum(g) / len(g)) if g else None\n    res["s_per_case_median"] = sorted(x["s"] for x in rows)[len(rows) // 2] if rows else None\n    if cap["audio"]:\n        try:\n            import io\n\n            import soundfile as sf\n            from datasets import load_dataset\n            ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")\n            a = ds[0]["audio"]; buf = io.BytesIO(); sf.write(buf, a["array"], a["sampling_rate"], format="WAV", subtype="PCM_16")\n            res["audio_probe"] = {"reference": ds[0]["text"], "transcript": ask(eng,\n                [L.Content.AudioBytes(buf.getvalue()), L.Content.Text("Transcribe exactly what is said. Output only the transcript.")], max_tok=120)}\n        except Exception as e:  # noqa: BLE001\n            res["audio_probe"] = f"error: {type(e).__name__}: {e}"\n    del eng\n    REPORT[tag] = res; save()\n    print(tag, json.dumps({k: res[k] for k in ["load_s", "capabilities", "text_probe", "analysis", "image_ablation_changed", "s_per_case_median"]}), flush=True)\n    return res\n\n\ndef module_diff():\n    """Which parameter groups did the fine-tune change? (merged vs base, max |Δ|) — decides whether sections of the\n    official .litertlm (vision/audio encoders) are valid for the fine-tuned model."""\n    from safetensors import safe_open\n    import torch\n    base_dir = snapshot_download("unsloth/gemma-4-E2B-it", local_dir="/tmp/gemma4_e2b")\n    def index(d):\n        out = {}\n        for f in glob.glob(f"{d}/*.safetensors"):\n            with safe_open(f, "pt") as h:\n                for k in h.keys():\n                    out[k] = f\n        return out\n    bi, mi = index(base_dir), index(SRC)\n    norm = lambda k: k.replace("base_model.model.", "")\n    groups = {}\n    for k, f in mi.items():\n        kb = norm(k)\n        if kb not in bi:\n            groups.setdefault("_missing_in_base", []).append(kb); continue\n        g = ("decoder_layers" if ".layers." in kb and "language_model" in kb else\n             kb.split(".")[2] if kb.startswith("model.") and len(kb.split(".")) > 2 else kb.split(".")[0])\n        with safe_open(f, "pt") as h1, safe_open(bi[kb], "pt") as h2:\n            a, b = h1.get_tensor(k).float(), h2.get_tensor(kb).float()\n        d = float((a - b).abs().max()) if a.shape == b.shape else float("inf")\n        groups.setdefault(g, []).append(d)\n    return {g: ({"n": len(v), "max_abs_diff": max(v), "changed": sum(x > 1e-3 for x in v)} if g != "_missing_in_base" else v[:20])\n            for g, v in groups.items()}\n\n\nREPORT["capabilities"] = caps(dst); save(); print("capabilities", REPORT["capabilities"], flush=True)\nlval = {json.loads(line)["id"]: json.loads(line) for line in open(f"{DATA}/llm_val.jsonl")}\nhf = None\nif MODE == "merged":\n    try:\n        REPORT["module_diff"] = module_diff(); save(); print("module_diff", json.dumps(REPORT["module_diff"])[:1500], flush=True)\n    except Exception as e:  # noqa: BLE001\n        REPORT["module_diff"] = f"error: {type(e).__name__}: {e}"\n    hf = json.load(open(glob.glob("/kaggle/input/**/reports/full.json", recursive=True)[0]))\n    cases = [lval[x["id"]] for x in hf["L4_analysis_rows"]["E1"]]\nelse:\n    cases = [r for r in lval.values() if r["task"] == "T1"][:10]\nexp = run_suite(dst, "exported", cases, n_gray=16 if MODE == "merged" else 5)\nif hf:   # ---- L6 export parity (test.md §5): HF merged (E1, greedy) vs this .litertlm (greedy), same prompts\n    e1 = {x["id"]: x for x in hf["L4_analysis_rows"]["E1"]}\n    a, b = summarize(list(e1.values()), ANALYSIS_KEYS), exp["analysis"]\n    same_cats = sum(set(x["cats"]) == set(e1[x["id"]]["cats"]) for x in exp["rows"]) / len(exp["rows"])\n    hf_abl = hf.get("image_ablation", {}).get("changed_rate")\n    L6 = {"hf": a, "litertlm": b, "json_validity_drop": round(a["valid"] - b["valid"], 4), "category_agreement_drop": round(a["cat_agree"] - b["cat_agree"], 4),\n          "category_sets_identical": round(same_cats, 4), "tier_ok": b["tier_ok"], "image_ablation_hf": hf_abl,\n          "image_ablation_litertlm": exp["image_ablation_changed"], "starts_with_brace": sum(x["out"].lstrip().startswith("{") for x in exp["rows"]) / len(exp["rows"])}\n    L6["pass"] = bool(L6["json_validity_drop"] <= 0.02 and L6["category_agreement_drop"] <= 0.03 and L6["tier_ok"] == 1.0\n                      and (not hf_abl or (exp["image_ablation_changed"] or 0) >= 0.75 * hf_abl) and L6["starts_with_brace"] == 1.0)\n    REPORT["L6"] = L6; save(); print("L6", json.dumps(L6), flush=True)\ntry:\n    ref = hf_hub_download("litert-community/gemma-4-E2B-it-litert-lm", "gemma-4-E2B-it.litertlm",\n                          revision="b3ca0d2f076785a8f4b2219ddbd2bdb99954eae1", local_dir="/tmp/ref")\n    run_suite(ref, "official_stock", cases[:10], n_gray=5)\nexcept Exception as e:  # noqa: BLE001\n    REPORT["official_stock"] = f"error: {type(e).__name__}: {e}"; save()\nprint("EXPORT_SUITE_DONE", flush=True)\n')
open('/tmp/export_patched.py', 'w').write('"""`litert-torch export_hf` (0.9.4) with three memory fixes for Gemma 4 E2B on Kaggle\'s 32 GB CPU machine.\n\nExport v4–v6 were OOM-killed (rc -9). v6 (with --experimental_lightweight_conversion) finished the text model and the\nembedder, then died in "Export per_layer_embedder model > Create MLIR Module" (min. MemAvailable seen 5.0 GB of 32 GB):\n1. upstream `export_additional_models_impl` calls `converter.convert(strict_export=False)` WITHOUT the lightweight flag,\n   although that table (262,144 x 8,960 fp32 = 9.4 GB) is traced for two signatures → pass the flag through;\n2. after the text prefill/decode model is written, the decoder-layer and audio-tower weights are not used by the\n   remaining steps (embedder, per-layer embedder, vision encoder, tokenizer) → release them (~2 B params, ~8 GB fp32).\nv7 then got through the per-layer embedder and died in "Export vision encoder models > vision_adapter > Create MLIR\nModule" (min. MemAvailable 1.4 GB): the 9.4 GB per-layer table and the 1.6 GB token embedding were still held →\n3. after the per-layer embedder is written, release the text embeddings too (the vision step uses only vision_tower +\n   embed_vision), and hand freed heap back to the OS (glibc malloc_trim) after each release.\nNothing about the exported graphs changes. Usage: python export_patched.py export_hf <src> <out> --flags...\n"""\nimport ctypes\nimport dataclasses\nimport gc\nimport os\n\nimport torch\nfrom litert_torch import cli\nfrom litert_torch.generative.export_hf.core import export_lib as E\n\n\ndef _additional_impl(name, exportable_module_cls, source_model_artifacts, export_config, exported_model_artifacts):\n    # upstream body, plus lightweight_conversion\n    model = source_model_artifacts.model\n    module = exportable_module_cls(model)\n    converter = E.converter_utils.Converter()\n    for signature_name, (sample_inputs, _) in module.get_sample_inputs(source_model_artifacts.text_model_config, export_config).items():\n        converter.add_signature(signature_name, module.eval(), sample_kwargs=sample_inputs)\n    lrt_model = converter.convert(lightweight_conversion=export_config.experimental_lightweight_conversion, strict_export=False)\n    model_path = os.path.join(export_config.work_dir, f"{name}.tflite")\n    lrt_model.export(model_path)\n    del lrt_model, converter\n    gc.collect()\n    recipe_list = export_config.quantization_recipe.split(",") if export_config.quantization_recipe else [None]\n    for recipe in recipe_list:\n        model_path = E.maybe_quantize_model(model_path, recipe)\n        gc.collect()\n    additional = exported_model_artifacts.additional_model_paths or {}\n    additional[name] = model_path\n    return dataclasses.replace(exported_model_artifacts, additional_model_paths=additional)\n\n\ndef _release(modules, label):\n    n = 0\n    for mod in modules:\n        if mod is None:\n            continue\n        for p in mod.parameters():\n            n += p.numel()\n            p.data = torch.empty(0, dtype=p.dtype)\n    gc.collect()\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except OSError:\n        pass\n    print(f"[skinnova] released {n / 1e9:.2f} B parameters after the {label}", flush=True)\n\n\n_text_export = E.export_text_prefill_decode_model\n_additional_export = E.export_additional_models\n\n\ndef _text_then_release(source_model_artifacts, export_config, exported_model_artifacts):\n    out = _text_export(source_model_artifacts, export_config, exported_model_artifacts)\n    m = source_model_artifacts.model.model\n    _release((m.language_model.layers, getattr(m, "audio_tower", None), getattr(m, "embed_audio", None)), "text export")\n    return out\n\n\ndef _additional_then_release(source_model_artifacts, export_config, exported_model_artifacts):\n    out = _additional_export(source_model_artifacts, export_config, exported_model_artifacts)\n    lm = source_model_artifacts.model.model.language_model\n    _release([getattr(lm, k, None) for k in ("embed_tokens_per_layer", "per_layer_model_projection", "embed_tokens")],\n             "per-layer embedder export")\n    return out\n\n\nE.export_additional_models_impl = _additional_impl\nE.export_text_prefill_decode_model = _text_then_release\nE.export_additional_models = _additional_then_release\n\nif __name__ == "__main__":\n    cli.main()\n')


In [ ]:
import subprocess, sys
p = subprocess.Popen(['/tmp/ve/bin/python', '/tmp/run.py', MODE], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
rc = p.wait(); print('exit code', rc)
assert rc == 0, 'export suite failed — see /kaggle/working/reports'
